# AutoML · M00: Índice Pre-Modelado

**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Institución** | UOC + Universitat Jaume I |
| **Email** | mjmorteruiz@uoc.edu · morte@uji.es |
| **Fase** | AutoML — Pre-Modelado |
| **Módulo** | M00 — Índice |

---

## 🎯 Qué hace

Genera la página índice HTML de la fase AutoML con la lista de módulos y su descripción.

## 📋 Requisitos

- `src/html/render.py` con `render_pagina_desde_fichero()`
- `src/html/navegacion.py` actualizado con fase_automl

## 📤 Genera

| Archivo | Contenido |
|---|---|
| `docs/html/fase_automl/fase_automl_index.html` | Índice navegable de la fase AutoML |

## 🔄 Flujo

```
render_pagina_desde_fichero() → HTML con navegación → escribe en disco
```

## ➡️ Siguiente

`fautoml_m01_baselines.ipynb` — modelos baseline de referencia


In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN
# ============================================================================

import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'src').exists():
        break
    ROOT = ROOT.parent


sys.path.insert(0, str(ROOT))
import pandas as pd
from src.config import RUTA_AUTOML, RUTA_HTML, info_entorno
from src.utils import crear_directorios, formato_numero_es
from src.html import generar_kpis_html, generar_tarjetas_html, generar_seccion_html
from src.html.render import render_pagina

RUTA_FASE_AUTOML_HTML = RUTA_HTML / 'fase_automl'
crear_directorios([RUTA_FASE_AUTOML_HTML])
fmt = formato_numero_es
info_entorno()

✓ Directorios verificados: 1
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw\datos_proyecto_sin_preinscrip.xlsx
✓

In [2]:
# ============================================================================
# CELDA 2: CARGAR DATOS + GENERAR HTML
# ============================================================================
# Solo dataset canónico: dataset_final_tfm.parquet (24 features).
# Módulos actualizados: M01-M07 con TabPFN v2 en M06.
# ============================================================================

# Cargar dataset canónico
ruta_ds = RUTA_AUTOML / 'dataset_final_tfm.parquet'
if ruta_ds.exists():
    df_ds    = pd.read_parquet(ruta_ds)
    n_reg    = len(df_ds)
    n_feat   = df_ds.shape[1] - 1
    pct_ab   = (df_ds['abandono']==1).mean()*100
    del df_ds
else:
    n_reg = 33621; n_feat = 24; pct_ab = 29.2

# Cargar top modelos si existe (M07)
ruta_top = RUTA_AUTOML / 'automl_top_modelos.parquet'
mejor_f1  = '—'
mejor_mod = '—'
if ruta_top.exists():
    df_top   = pd.read_parquet(ruta_top)
    mejor    = df_top.sort_values('f1', ascending=False).iloc[0]
    mejor_f1 = f'{mejor["f1"]:.4f}'
    mejor_mod = mejor['model_name'][:20]

KPIS = [
    {'valor': fmt(n_reg),    'titulo': 'Expedientes'},
    {'valor': str(n_feat),   'titulo': 'Features'},
    {'valor': f'{pct_ab:.1f}%', 'titulo': 'Abandono'},
    {'valor': mejor_f1,      'titulo': f'Mejor F1 ({mejor_mod})'},
]

MODULOS = [
    {'archivo': 'm01_baselines.html',   'emoji': '📊', 'titulo': 'M01: Baselines',
     'desc': 'LogReg, RF, GBM, XGBoost con CV=5. Baseline de referencia.', 'color': '#3182ce'},
    {'archivo': 'm02_lazypredict.html', 'emoji': '⚡', 'titulo': 'M02: LazyPredict',
     'desc': 'Screening ~30 modelos sin tuning. Orientativo.', 'color': '#d69e2e'},
    {'archivo': 'm03_pycaret.html',     'emoji': '🤖', 'titulo': 'M03: PyCaret',
     'desc': 'AutoML con CV=5. ~15 algoritmos comparados.', 'color': '#38a169'},
    {'archivo': 'm04_h2o.html',         'emoji': '💧', 'titulo': 'M04: H2O',
     'desc': 'AutoML distribuido + StackedEnsemble. Java requerido.', 'color': '#805ad5'},
    {'archivo': 'm05_autogluon.html',   'emoji': '🚀', 'titulo': 'M05: AutoGluon',
     'desc': 'Multi-layer stacking automático. Mejor ensemble.', 'color': '#ed8936'},
    {'archivo': 'm06_tabpfn.html',      'emoji': '🧠', 'titulo': 'M06: TabPFN v2',
     'desc': 'Transformer preentrenado. Estado del arte tabular. Sin hiperparámetros.',
     'color': '#e53e3e'},
    {'archivo': 'm07_comparativa.html', 'emoji': '🏆', 'titulo': 'M07: Comparativa Final',
     'desc': 'Ranking global M01-M06. Baseline definitivo para Fase 5.', 'color': '#2d3748'},
]

tarjetas = [{'titulo': m['titulo'], 'descripcion': m['desc'], 'emoji': m['emoji'],
             'link': m['archivo'], 'link_texto': 'Abrir →', 'color': m['color']}
            for m in MODULOS]

s1 = generar_seccion_html('Resumen', f'''
<p>Screening de <strong>6 frameworks AutoML</strong> sobre <strong>{fmt(n_reg)} expedientes</strong>
con <strong>24 features auditadas</strong> (dataset_final_tfm.parquet, D_strict).</p>
<div style="background:#f0fff4;padding:15px;border-radius:8px;margin-top:15px;border-left:4px solid #38a169;">
  <strong>✅ Dataset único:</strong> dataset_final_tfm.parquet — 24 features sin leakage,
  auditadas en Fase 3. Un solo dataset canónico para todos los frameworks.
</div>
<div style="background:#ebf8ff;padding:15px;border-radius:8px;margin-top:10px;border-left:4px solid #3182ce;">
  <strong>📌 Entorno único:</strong> tfm_abandono — todos los frameworks en el mismo entorno.
  LazyPredict, PyCaret, H2O, AutoGluon y TabPFN v2 instalados directamente.
</div>''', '⚡')

ruta_html = RUTA_FASE_AUTOML_HTML / 'fase_automl_index.html'
render_pagina(
    'fautoml_m00_indice.ipynb',
    s1 + generar_kpis_html(KPIS) + generar_seccion_html('Módulos', generar_tarjetas_html(tarjetas), '📦'),
    ruta_html,
    carpeta_notebook='fase_automl'
)
print(f'HTML: {ruta_html}')


HTML: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase_automl\fase_automl_index.html


In [3]:
# ============================================================================
# CELDA 3: RESUMEN
# ============================================================================

print('\n' + '='*60)
print('✅ AUTOML-M00 COMPLETADO')
print('='*60)
print(f'Archivo: {ruta_html}')
print(f'Expedientes: {fmt(n_reg)}')
print(f'Features: {n_feat} (dataset_final_tfm.parquet)')
print(f'Target: {pct_ab:.1f}% abandono')
print(f'Frameworks: M01 Baselines → M07 Comparativa')
print(f'\n🎯 Siguiente: fautoml_m01_baselines.ipynb')
print('='*60)



✅ AUTOML-M00 COMPLETADO
Archivo: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase_automl\fase_automl_index.html
Expedientes: 33.621
Features: 24 (dataset_final_tfm.parquet)
Target: 29.2% abandono
Frameworks: M01 Baselines → M07 Comparativa

🎯 Siguiente: fautoml_m01_baselines.ipynb
